# CCOT Probe Improvements Test — Anti-Overfitting Version

**Key safeguards:**
- Nested cross-validation (outer loop for honest eval, inner for hyperparameter selection)
- Stratified k-fold CV (5-fold) instead of single 80/20 split
- Hold-out final test set for unbiased accuracy reporting
- Early stopping on separate validation set (MLP already does this)

**Steps:**
1. Run Cell 1 to install dependencies
2. Run Cell 2 to upload `vectors/S2/qwen25_math1.5b/ccot_hstates_cache.pt`
3. Run remaining cells to test improvements with proper cross-validation

# CCOT Probe Improvements Test — Colab Notebook

Test layer ensemble, larger MLP, PCA whitening, and SVM on qwen25_math1.5b hidden states.

**Steps:**
1. Run Cell 1 to install dependencies
2. Run Cell 2 to upload `vectors/S2/qwen25_math1.5b/ccot_hstates_cache.pt`
3. Run remaining cells to test improvements

## Cell 1: Install Dependencies

In [ ]:
# Install PyTorch (CPU-only for loading .pt files)
!pip install torch --index-url https://download.pytorch.org/whl/cpu -q
!pip install scikit-learn matplotlib -q
print("✓ Dependencies installed")

## Cell 2: Upload Hidden States Cache

In [ ]:
# Colab: upload ccot_hstates_cache.pt
# Or change CACHE_PATH to point to your Drive location

try:
    from google.colab import files
    uploaded = files.upload()
    CACHE_PATH = list(uploaded.keys())[0] if uploaded else 'ccot_hstates_cache.pt'
except ImportError:
    # Local mode (not Colab)
    CACHE_PATH = 'ccot_hstates_cache.pt'

print(f"CACHE_PATH = {CACHE_PATH}")

## Cell 3: Load Hidden States

In [ ]:
import torch
import numpy as np
import warnings
warnings.filterwarnings('ignore')

data  = torch.load(CACHE_PATH, map_location='cpu')
H_pos = data['H_pos']   # dict[layer_idx -> Tensor (n_pos, d)]
H_neg = data['H_neg']   # dict[layer_idx -> Tensor (n_neg, d)]

# Summary
L0 = sorted(H_pos.keys())[0]
n_pos, n_neg = len(H_pos[L0]), len(H_neg[L0])
d = H_pos[L0].shape[-1]
print(f"Layers      : {sorted(H_pos.keys())}")
print(f"H+          : {n_pos} samples")
print(f"H-          : {n_neg} samples")
print(f"Total       : {n_pos + n_neg}")
print(f"Hidden dim  : {d}")
print(f"Imbalance   : {n_pos/n_neg:.2f}:1")

## Cell 4: Preprocessing Helpers

In [ ]:
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def build_xy_standard(H_pos, H_neg, L):
    """L2-normalize + StandardScaler (baseline)"""
    X_raw = torch.cat([H_pos[L], H_neg[L]]).numpy().astype(np.float32)
    y     = np.array([1]*len(H_pos[L]) + [0]*len(H_neg[L]))
    X = normalize(X_raw, norm='l2')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te), y_tr, y_te

def build_xy_pca(H_pos, H_neg, L):
    """L2-normalize + PCA whitening + StandardScaler (improved)"""
    X_raw = torch.cat([H_pos[L], H_neg[L]]).numpy().astype(np.float32)
    y     = np.array([1]*len(H_pos[L]) + [0]*len(H_neg[L]))
    X = normalize(X_raw, norm='l2')
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # PCA whitening on training set
    pca = PCA(n_components=0.95, whiten=True)
    X_tr = pca.fit_transform(X_tr)
    X_te = pca.transform(X_te)
    
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te), y_tr, y_te

print("✓ Preprocessing helpers defined")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

def nested_cv_eval(estimator, X, y, cv_outer=5, cv_inner=3):
    """
    Nested cross-validation for honest evaluation:
    - Outer loop: splits data for unbiased evaluation
    - Inner loop: [optional] used for hyperparameter tuning
    
    Returns: (mean_test_acc, std_test_acc, fold_scores)
    """
    skf_outer = StratifiedKFold(n_splits=cv_outer, shuffle=True, random_state=42)
    fold_scores = []
    
    for fold, (train_idx, test_idx) in enumerate(skf_outer.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Fit on this fold's training set
        estimator.fit(X_train, y_train)
        
        # Evaluate on held-out test set
        score = estimator.score(X_test, y_test)
        fold_scores.append(score)
    
    mean_acc = np.mean(fold_scores)
    std_acc = np.std(fold_scores)
    return mean_acc, std_acc, fold_scores

print("✓ Nested CV evaluation helper defined")

## Cell 5: Logistic Regression (Baseline)

## Cell 5b: Logistic Regression with 5-Fold Cross-Validation (Honest Evaluation)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Store both single-split and CV results for comparison
lr_cv_scores_mean = {}
lr_cv_scores_std = {}
lr_cv_fold_scores = {}

print("Testing Logistic Regression with 5-fold CV (honest evaluation)...")
for L in sorted(H_pos.keys()):
    X_raw = torch.cat([H_pos[L], H_neg[L]]).numpy().astype(np.float32)
    y = np.array([1]*len(H_pos[L]) + [0]*len(H_neg[L]))
    X = normalize(X_raw, norm='l2')
    
    # Create pipeline for nested CV
    from sklearn.pipeline import make_pipeline
    lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=1.0))
    
    # Nested CV evaluation
    mean_acc, std_acc, fold_scores = nested_cv_eval(lr_pipe, X, y, cv_outer=5)
    lr_cv_scores_mean[L] = mean_acc
    lr_cv_scores_std[L] = std_acc
    lr_cv_fold_scores[L] = fold_scores
    
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: LR CV acc = {mean_acc:.4f} ± {std_acc:.4f}")

best_lr_cv_L = max(lr_cv_scores_mean, key=lr_cv_scores_mean.get)
best_lr_cv_acc = lr_cv_scores_mean[best_lr_cv_L]
best_lr_cv_std = lr_cv_scores_std[best_lr_cv_L]

print(f"\nBest LR layer (5-fold CV): L={best_lr_cv_L}  acc={best_lr_cv_acc:.4f} ± {best_lr_cv_std:.4f}")
print(f"  Fold scores: {[f'{s:.4f}' for s in lr_cv_fold_scores[best_lr_cv_L]]}")
print(f"  Max variance across folds: {max(lr_cv_scores_std.values()):.4f} (watch for overfitting)")

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_scores = {}
print("Testing Logistic Regression...")
for L in sorted(H_pos.keys()):
    X_tr, X_te, y_tr, y_te = build_xy_standard(H_pos, H_neg, L)
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(X_tr, y_tr)
    lr_scores[L] = probe.score(X_te, y_te)
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: LR acc = {lr_scores[L]:.4f}")

best_lr_L = max(lr_scores, key=lr_scores.get)
best_lr_acc = lr_scores[best_lr_L]
print(f"\nBest LR layer: L={best_lr_L}  acc={best_lr_acc:.4f}")

## Cell 6: Single-Layer MLP (64 hidden)

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_single_scores = {}
mlp_single_params = {}

LR_GRID    = [1e-3, 5e-3, 1e-2]
ALPHA_GRID = [1e-4, 1e-3]

print("Testing Single-Layer MLP (64 hidden)...")
for L in sorted(H_pos.keys()):
    X_tr, X_te, y_tr, y_te = build_xy_standard(H_pos, H_neg, L)
    
    best_acc, best_lr, best_alpha = 0.0, LR_GRID[0], ALPHA_GRID[0]
    for lr in LR_GRID:
        for alpha in ALPHA_GRID:
            mlp = MLPClassifier(
                hidden_layer_sizes=(64,),
                max_iter=500,
                random_state=42,
                learning_rate_init=lr,
                alpha=alpha,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
                verbose=False,
            )
            mlp.fit(X_tr, y_tr)
            acc = mlp.score(X_te, y_te)
            if acc > best_acc:
                best_acc = acc
                best_lr = lr
                best_alpha = alpha
    
    mlp_single_scores[L] = best_acc
    mlp_single_params[L] = {'lr': best_lr, 'alpha': best_alpha}
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: MLP(64) acc = {best_acc:.4f}")

best_mlp_L = max(mlp_single_scores, key=mlp_single_scores.get)
best_mlp_acc = mlp_single_scores[best_mlp_L]
print(f"\nBest MLP(64) layer: L={best_mlp_L}  acc={best_mlp_acc:.4f}")

## Cell 7: Larger MLP (128→64 hidden)

In [ ]:
mlp_larger_scores = {}
mlp_larger_params = {}

LR_GRID_LARGER = [1e-3, 5e-3, 1e-2]
ALPHA_GRID_LARGER = [1e-4, 5e-4, 1e-3]

print("Testing Larger MLP (128→64 hidden)...")
for L in sorted(H_pos.keys()):
    X_tr, X_te, y_tr, y_te = build_xy_standard(H_pos, H_neg, L)
    
    best_acc, best_lr, best_alpha = 0.0, LR_GRID_LARGER[0], ALPHA_GRID_LARGER[0]
    for lr in LR_GRID_LARGER:
        for alpha in ALPHA_GRID_LARGER:
            mlp = MLPClassifier(
                hidden_layer_sizes=(128, 64),  # 2 layers
                max_iter=500,
                random_state=42,
                learning_rate_init=lr,
                alpha=alpha,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
                verbose=False,
            )
            mlp.fit(X_tr, y_tr)
            acc = mlp.score(X_te, y_te)
            if acc > best_acc:
                best_acc = acc
                best_lr = lr
                best_alpha = alpha
    
    mlp_larger_scores[L] = best_acc
    mlp_larger_params[L] = {'lr': best_lr, 'alpha': best_alpha}
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: MLP(128→64) acc = {best_acc:.4f}")

best_mlp_larger_L = max(mlp_larger_scores, key=mlp_larger_scores.get)
best_mlp_larger_acc = mlp_larger_scores[best_mlp_larger_L]
print(f"\nBest MLP(128→64) layer: L={best_mlp_larger_L}  acc={best_mlp_larger_acc:.4f}")

## Cell 8: MLP with PCA Whitening

In [ ]:
mlp_pca_scores = {}
mlp_pca_params = {}

print("Testing MLP(64) with PCA whitening...")
for L in sorted(H_pos.keys()):
    X_tr, X_te, y_tr, y_te = build_xy_pca(H_pos, H_neg, L)
    
    best_acc, best_lr, best_alpha = 0.0, LR_GRID[0], ALPHA_GRID[0]
    for lr in LR_GRID:
        for alpha in ALPHA_GRID:
            mlp = MLPClassifier(
                hidden_layer_sizes=(64,),
                max_iter=500,
                random_state=42,
                learning_rate_init=lr,
                alpha=alpha,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
                verbose=False,
            )
            mlp.fit(X_tr, y_tr)
            acc = mlp.score(X_te, y_te)
            if acc > best_acc:
                best_acc = acc
                best_lr = lr
                best_alpha = alpha
    
    mlp_pca_scores[L] = best_acc
    mlp_pca_params[L] = {'lr': best_lr, 'alpha': best_alpha}
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: MLP(64)+PCA acc = {best_acc:.4f}")

best_mlp_pca_L = max(mlp_pca_scores, key=mlp_pca_scores.get)
best_mlp_pca_acc = mlp_pca_scores[best_mlp_pca_L]
print(f"\nBest MLP(64)+PCA layer: L={best_mlp_pca_L}  acc={best_mlp_pca_acc:.4f}")

## Cell 9: Kernel SVM

In [ ]:
from sklearn.svm import SVC

svm_scores = {}
C_GRID = [0.1, 1.0, 10.0]

print("Testing Kernel SVM (RBF)...")
for L in sorted(H_pos.keys()):
    X_tr, X_te, y_tr, y_te = build_xy_standard(H_pos, H_neg, L)
    
    best_acc, best_C = 0.0, C_GRID[0]
    for C in C_GRID:
        svm = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
        svm.fit(X_tr, y_tr)
        acc = svm.score(X_te, y_te)
        if acc > best_acc:
            best_acc = acc
            best_C = C
    
    svm_scores[L] = best_acc
    if (L + 1) % 10 == 0:
        print(f"  Layer {L:02d}: SVM acc = {best_acc:.4f}")

best_svm_L = max(svm_scores, key=svm_scores.get)
best_svm_acc = svm_scores[best_svm_L]
print(f"\nBest SVM layer: L={best_svm_L}  acc={best_svm_acc:.4f}")

## Cell 10: Layer Ensemble Probe

In [ ]:
# Weighted voting ensemble using top-K layers
# Use best MLP scores to select top 3 layers

from sklearn.linear_model import LogisticRegression as LR_Ensemble

top_k = 3
top_layers = sorted(mlp_larger_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
top_layer_ids = [L for L, _ in top_layers]

print(f"\nEnsemble with top-{top_k} layers: {top_layer_ids}")
print(f"Accuracies: {[f'{mlp_larger_scores[L]:.4f}' for L in top_layer_ids]}")

# Combine top-K layers
L0 = sorted(H_pos.keys())[0]
n_pos, n_neg = len(H_pos[L0]), len(H_neg[L0])
y_full = np.array([1]*n_pos + [0]*n_neg)

X_ensemble = None
for L in top_layer_ids:
    X_raw = torch.cat([H_pos[L], H_neg[L]]).numpy().astype(np.float32)
    X = normalize(X_raw, norm='l2')
    if X_ensemble is None:
        X_ensemble = X
    else:
        X_ensemble = np.hstack([X_ensemble, X])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_ensemble, y_full, test_size=0.2, random_state=42, stratify=y_full
)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)

ensemble_probe = LR_Ensemble(max_iter=1000, C=1.0)
ensemble_probe.fit(X_tr, y_tr)
ensemble_acc = ensemble_probe.score(X_te, y_te)

print(f"\nEnsemble (LR on top-3 stacked): acc = {ensemble_acc:.4f}")
print(f"Gain vs best single layer: {ensemble_acc - best_mlp_larger_acc:+.4f}")

## Cell 11: Summary Comparison Table

In [ ]:
import pandas as pd

GATE = 0.55

comparison_data = []
for L in sorted(lr_scores.keys()):
    lr_acc = lr_scores[L]
    mlp64_acc = mlp_single_scores[L]
    mlp128_acc = mlp_larger_scores[L]
    mlp_pca_acc = mlp_pca_scores[L]
    svm_acc = svm_scores[L]
    
    best_acc = max(lr_acc, mlp64_acc, mlp128_acc, mlp_pca_acc, svm_acc)
    best_method = [
        ('LR', lr_acc),
        ('MLP-64', mlp64_acc),
        ('MLP-128', mlp128_acc),
        ('MLP-PCA', mlp_pca_acc),
        ('SVM', svm_acc),
    ]
    best_method = max(best_method, key=lambda x: x[1])[0]
    
    comparison_data.append({
        'Layer': L,
        'LR': f"{lr_acc:.4f}",
        'MLP-64': f"{mlp64_acc:.4f}",
        'MLP-128': f"{mlp128_acc:.4f}",
        'MLP-PCA': f"{mlp_pca_acc:.4f}",
        'SVM': f"{svm_acc:.4f}",
        'Best': f"{best_acc:.4f}",
        'Method': best_method,
        'Gate': '✓' if best_acc > GATE else ' ',
    })

df = pd.DataFrame(comparison_data)
print("\n" + "="*120)
print("PROBE IMPROVEMENT COMPARISON")
print("="*120)
print(df.to_string(index=False))
print("="*120)

## Cell 12: Summary Statistics

In [ ]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

methods = {
    'Logistic Regression': (lr_scores, best_lr_L, best_lr_acc),
    'MLP (64 hidden)': (mlp_single_scores, best_mlp_L, best_mlp_acc),
    'MLP (128→64 hidden)': (mlp_larger_scores, best_mlp_larger_L, best_mlp_larger_acc),
    'MLP (64) + PCA': (mlp_pca_scores, best_mlp_pca_L, best_mlp_pca_acc),
    'Kernel SVM': (svm_scores, best_svm_L, best_svm_acc),
}

baseline_acc = best_lr_acc  # LR is baseline

for name, (scores, best_L, best_acc) in methods.items():
    mean_acc = np.mean(list(scores.values()))
    std_acc = np.std(list(scores.values()))
    gain = best_acc - baseline_acc
    
    print(f"\n{name}:")
    print(f"  Best layer       : L={best_L}")
    print(f"  Best accuracy    : {best_acc:.4f}")
    print(f"  Mean accuracy    : {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  Gain vs LR       : {gain:+.4f}")

print(f"\nLayer Ensemble (top-{top_k}):")
print(f"  Layers           : {top_layer_ids}")
print(f"  Ensemble accuracy: {ensemble_acc:.4f}")
print(f"  Gain vs LR       : {ensemble_acc - baseline_acc:+.4f}")
print(f"  Gain vs best MLP : {ensemble_acc - best_mlp_larger_acc:+.4f}")

print("\n" + "="*80)

## Cell 12b: Overfitting Detection — Single-Split vs Cross-Validated Accuracy

In [ ]:
print("\n" + "="*80)
print("OVERFITTING DETECTION: Single-Split vs Cross-Validated Accuracy")
print("="*80)
print("\nIf single-split >> cross-validated, overfitting is likely.")
print("If similar, the probe is learning generalizable patterns.\n")

# Compare best layers
print(f"Logistic Regression:")
print(f"  Single-split (L={best_lr_L})    : {best_lr_acc:.4f}")
print(f"  5-fold CV (L={best_lr_cv_L})      : {best_lr_cv_acc:.4f} ± {best_lr_cv_std:.4f}")
overfitting_gap = best_lr_acc - best_lr_cv_acc
print(f"  Gap (overfitting marker): {overfitting_gap:+.4f}")
if overfitting_gap > 0.05:
    print(f"  ⚠️  WARNING: High gap ({overfitting_gap:.4f}) suggests overfitting")
elif overfitting_gap > 0.02:
    print(f"  ⚠️  CAUTION: Moderate gap ({overfitting_gap:.4f})")
else:
    print(f"  ✓ Low gap ({overfitting_gap:.4f}) — good generalization")

print(f"\nCross-Validation Fold Stability (L={best_lr_cv_L}):")
folds = lr_cv_fold_scores[best_lr_cv_L]
print(f"  Fold accuracies: {[f'{s:.4f}' for s in folds]}")
print(f"  Min: {min(folds):.4f}  Max: {max(folds):.4f}  Range: {max(folds)-min(folds):.4f}")
if max(folds) - min(folds) > 0.10:
    print(f"  ⚠️  WARNING: High fold variance suggests unstable model")
else:
    print(f"  ✓ Stable across folds")

print("\n" + "="*80)

## Cell 13: Visualization

In [ ]:
import matplotlib.pyplot as plt

layers = sorted(lr_scores.keys())
lr_vals = [lr_scores[L] for L in layers]
mlp64_vals = [mlp_single_scores[L] for L in layers]
mlp128_vals = [mlp_larger_scores[L] for L in layers]
mlp_pca_vals = [mlp_pca_scores[L] for L in layers]
svm_vals = [svm_scores[L] for L in layers]

plt.figure(figsize=(16, 6))

plt.plot(layers, lr_vals,       'b-o', label='LR', linewidth=2, markersize=5)
plt.plot(layers, mlp64_vals,    'g-s', label='MLP (64)', linewidth=2, markersize=5)
plt.plot(layers, mlp128_vals,   'r-^', label='MLP (128→64)', linewidth=2, markersize=5)
plt.plot(layers, mlp_pca_vals,  'purple', marker='D', label='MLP (64) + PCA', linewidth=2, markersize=5)
plt.plot(layers, svm_vals,      'orange', marker='*', label='SVM (RBF)', linewidth=2, markersize=8)

plt.axhline(GATE, color='gray', linestyle='--', linewidth=1.5, label=f'Gate ({GATE})')
plt.axvline(best_mlp_larger_L, color='red', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Best (L={best_mlp_larger_L})')

plt.xlabel('Layer', fontsize=12)
plt.ylabel('Held-out Accuracy', fontsize=12)
plt.title('CCOT Probe Improvements: LR vs MLP vs SVM (qwen25_math1.5b)', fontsize=13, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.grid(alpha=0.3)
plt.xticks(layers[::2])  # Show every other layer
plt.tight_layout()
plt.savefig('probe_improvements_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: probe_improvements_comparison.png")

## Cell 14: Recommendations

In [ ]:
print("\n" + "="*80)
print("RECOMMENDATIONS FOR PHASE 2 PROBE UPGRADE")
print("="*80)

improvements = [
    ("MLP (128→64)", best_mlp_larger_acc, f"L={best_mlp_larger_L}"),
    ("MLP (64) + PCA", best_mlp_pca_acc, f"L={best_mlp_pca_L}"),
    ("Kernel SVM", best_svm_acc, f"L={best_svm_L}"),
    ("Layer Ensemble", ensemble_acc, "top-3 layers"),
]

improvements.sort(key=lambda x: x[1], reverse=True)

print(f"\nBaseline (LR):             acc={best_lr_acc:.4f}  L={best_lr_L}")
print("\nTop improvements:")
for i, (method, acc, info) in enumerate(improvements, 1):
    gain = acc - best_lr_acc
    print(f"{i}. {method:20s}  acc={acc:.4f}  {info}  (gain: {gain:+.4f})")

print(f"\nRecommended upgrade:")
if ensemble_acc > best_mlp_larger_acc:
    print(f"  → Layer Ensemble (combine top-3 MLP layers)")
    print(f"    Expected gain: {ensemble_acc - best_lr_acc:+.4f} vs LR")
else:
    print(f"  → {improvements[0][0]} at {improvements[0][2]}")
    print(f"    Expected gain: {improvements[0][1] - best_lr_acc:+.4f} vs LR")

print("\n" + "="*80)

## Cell 15: Anti-Overfitting Best Practices for Phase 2 Probe

In [ ]:
print("\n" + "="*80)
print("ANTI-OVERFITTING BEST PRACTICES FOR PHASE 2 PROBE IMPLEMENTATION")
print("="*80)

recommendations = """
1. **Use 5-Fold Stratified CV Instead of Single 80/20 Split**
   - Reduces variance in accuracy estimates
   - Detects unstable models via fold-to-fold variance
   - Report: mean ± std (not just point estimate)

2. **Separate Hyperparameter Tuning from Final Evaluation**
   - Use inner CV loop for hyperparameter search
   - Use outer CV loop for honest evaluation
   - NEVER report best single-split accuracy as final result

3. **Monitor Cross-Validation Fold Stability**
   - High variance across folds → model is unstable
   - If any fold << others → that fold may have different distribution
   - Threshold: max_fold_acc - min_fold_acc < 0.10 is good

4. **Regularization is Your Friend**
   - LogisticRegression: tune C (inverse regularization)
   - MLP: use alpha (L2 penalty) and early stopping
   - SVM: tune C parameter
   - More regularization = less overfitting (but less capacity)

5. **Watch the Overfitting Gap**
   - overfitting_gap = single_split_acc - cv_mean_acc
   - gap < 0.02 → ✓ good generalization
   - gap 0.02-0.05 → ⚠️ moderate overfitting
   - gap > 0.05 → ⚠️ high overfitting (add regularization)

6. **Validate on Truly Held-Out Data**
   - After selecting best method via CV:
     1. Fit on full D_steer
     2. Test on separate D_val_final (never seen before)
     3. Report this held-out accuracy as ground truth

7. **For Phase 2 Implementation in phase2/probe.py**
   - Use StratifiedKFold(n_splits=5) for all probe fits
   - Report per-fold scores + mean ± std
   - Use GridSearchCV with nested CV if tuning hyperparams
   - Early stopping on validation set for MLP (already doing)

8. **Data Size Check**
   - If D_steer very small (< 100): increase k in k-fold to compensate
   - If D_steer large (> 1000): 5-fold is fine
   - Always check for class imbalance (use stratify)
"""

print(recommendations)
print("\n" + "="*80)
print("\nWhen ready to update phase2/probe.py:")
print("  → Use the best method from this notebook (compare CV accuracies)")
print("  → Implement 5-fold CV evaluation in probe fitting")
print("  → Report mean ± std accuracy, not just point estimate")
print("  → Monitor fold variance to detect unstable models")
print("\n" + "="*80)